# Fitting the NGC 4151 Spectrum with a Cutoff Power Law

## Introduction

This notebook demonstrates an unbinned spectral analysis of NGC 4151 using an injected cutoff power-law (CPL) spectrum.

> **Note on Underlying Mechanics:** To keep this tutorial concise, the detailed explanations of the underlying classes (such as `NFResponse`, `NFBackground`, and `UnbinnedThreeMLPointSourceResponseIRFAdaptive`) have been omitted here. If you are unfamiliar with the Neural-Network Response, the Background Approximation, or the corresponding hardware requirements introduced for DC4, **please refer to the foundational tutorial:** `example_grb_fit_normalizing_flows.ipynb`.

## Example

### Basic Setup

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [2]:
from pathlib import Path
from cosipy.spacecraftfile import SpacecraftHistory

import astropy.units as u
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np

from threeML import PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Cutoff_powerlaw

from cosipy.threeml.unbinned_model_folding import CachedUnbinnedThreeMLModelFolding
from cosipy.statistics import UnbinnedLikelihood
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.interfaces.expectation_interface import SumExpectationDensity

from cosipy.event_selection.time_selection import TimeSelector
from cosipy.data_io.EmCDSUnbinnedData import TimeTagEmCDSEventDataInSCFrameFromDC3Fits

from cosipy.response.ml.NFResponse import NFResponse
from cosipy.response.ml.nf_instrument_response_function import UnpolarizedNFFarFieldInstrumentResponseFunction

from cosipy.background_estimation.ml.NFBackground import NFBackground
from cosipy.background_estimation.ml.nf_unbinned_background import FreeNormNFUnbinnedBackground

from cosipy.threeml.ml.optimized_unbinned_folding import UnbinnedThreeMLPointSourceResponseIRFAdaptive

21:51:54 INFO      Starting 3ML!                                                                     ]8;id=7713009;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=7713010;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=7713016;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=7713017;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

         WARNING   but are inform you about optional packages that can be installed                  ]8;id=7713023;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=7713024;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#46\46]8;;\

         WARNING    to disable these messages, turn off start_warning in your config file            ]8;id=7713030;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=7713031;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#47\47]8;;\

         WARNING   no display variable set. using backend for graphics without display (agg)         ]8;id=7713037;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=7713038;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#53\53]8;;\

21:51:55 WARNING   ROOT minimizer not available                                                ]8;id=7713045;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=7713046;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1208\1208]8;;\

         WARNING   Multinest minimizer not available                                           ]8;id=7713052;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=7713053;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1218\1218]8;;\

         WARNING   PyGMO is not available                                                      ]8;id=7713059;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=7713060;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1228\1228]8;;\

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=7713066;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=7713067;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

21:51:56 WARNING   No fermitools installed                                              ]8;id=7713074;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=7713075;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

In [3]:
pointing_cut_path = Path(
    "/home/parshap/COSI/Radio_Quiet_AGN/DC4/FOV_Cut"
)
model_path = Path("/home/parshap/COSI/Radio_Quiet_AGN/DC4")
cache_path = Path(
    "/home/parshap/COSI/Radio_Quiet_AGN/DC4/unbinned_cache"
)

ngc4151_data_path = pointing_cut_path / (
    "NGC_4151_3months_unbinned_data_filtered_with_SAAcut_"
    "NGC4151_cut.fits.gz"
)
bkg_data_path = pointing_cut_path / (
    "Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_"
    "withSAAbck_NGC4151_cut.fits.gz"
)
sc_orientation_path = pointing_cut_path / (
    "DC4_final_530km_3_month_with_slew_15sbins_"
    "GalacticEarth_SAA_NGC4151_cut.fits"
)

rsp_path = model_path / "unpolarized_nfresponse_v1-01.pt"
bkg_path = model_path / "nfbackground_v1-01.pt"

Use the full time range of the pointing-cut orientation file.

In [4]:
sc_orientation = SpacecraftHistory.open(sc_orientation_path)
tstart = sc_orientation.tstart
tstop = sc_orientation.tstop

print(f"Observation interval: {tstart.isot} to {tstop.isot}")

Observation interval: 2028-03-01T14:18:45.000 to 2028-06-01T01:45:15.000


In [5]:
data_file = [ngc4151_data_path, bkg_data_path]
selector = TimeSelector(tstart = sc_orientation.tstart, tstop = sc_orientation.tstop)
data = TimeTagEmCDSEventDataInSCFrameFromDC3Fits(data_file, selection=selector)

In [6]:
print(f"This analysis uses {data.nevents} events")

This analysis uses 25125951 events


### Initializing Models and Folding Objects
Setting up the neural-network response (`NFResponse`), the background approximation (`NFBackground`), and the adaptive folding objects.

In [7]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. Run this notebook inside a GPU Slurm job."
    )

device_name = os.environ.get("COSI_DEVICE", "cuda:0")
device = torch.device(device_name)
if device.type != "cuda" or device.index is None:
    raise ValueError("COSI_DEVICE must be an indexed CUDA device, such as cuda:0")
if device.index >= torch.cuda.device_count():
    raise RuntimeError(
        f"Requested {device_name}, but only {torch.cuda.device_count()} GPU(s) are visible"
    )

devices = [device_name]
print(f"Visible GPUs: {torch.cuda.device_count()}")
print(f"Using {device_name}: {torch.cuda.get_device_name(device)}")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}")

Visible GPUs: 1
Using cuda:0: NVIDIA A100 80GB PCIe
PyTorch: 2.13.0+cu130; CUDA: 13.0


In [8]:
rsp = NFResponse(
    path_to_model=rsp_path,
    area_batch_size=300_000,
    density_batch_size=100_000, 
    devices=devices,
    area_compile_mode=None,
    density_compile_mode=None,
    show_progress=True)

irf = UnpolarizedNFFarFieldInstrumentResponseFunction(rsp)

In [9]:
bkg_model = NFBackground(
    path_to_model=bkg_path,
    density_batch_size=100_000,
    devices=devices,
    density_compile_mode=None,
    show_progress=True)

bkg = FreeNormNFUnbinnedBackground(
    model=bkg_model, 
    data=data, 
    sc_history=sc_orientation, 
    label="bkg_norm")

In [10]:
psr = UnbinnedThreeMLPointSourceResponseIRFAdaptive(
    data=data, 
    irf=irf, 
    sc_history=sc_orientation, 
    show_progress=True, 
    force_energy_node_caching=True, 
    reduce_memory=True) 

psr.cache_batch_size = 100_000
psr.integration_batch_size = 100_000

The CPL model uses `astromodels.Cutoff_powerlaw`, with the injected NGC 4151 parameters as the initial fit values.

In [ ]:
l = 155.07
b = 75.06

index = -1.75  # Injected value
piv = 1. * u.keV
xc = 200. * u.keV
K_thermal = 0.15 / u.cm / u.cm / u.s / u.keV

spectrum = Cutoff_powerlaw(
    index=index,
    K=K_thermal.value,
    piv=piv.value,
    xc=xc.value,
)

spectrum.index.min_value = -4
spectrum.index.max_value = 2
spectrum.K.min_value = 1e-5
spectrum.K.max_value = 1e2
spectrum.xc.min_value = 100
spectrum.xc.max_value = 10000

spectrum.K.unit = K_thermal.unit
spectrum.piv.unit = piv.unit
spectrum.xc.unit = xc.unit

spectrum.index.delta = 0.01
spectrum.K.delta = 0.01
spectrum.xc.delta = 10.

In [12]:
spectrum_inj = deepcopy(spectrum)

In [13]:
source = PointSource("NGC_4151",
                     l=l,
                     b=b,
                     spectral_shape=spectrum)
model = Model(source)

In [14]:
response = CachedUnbinnedThreeMLModelFolding(psr)

In [15]:
expectation_density = SumExpectationDensity(response, bkg)

In [16]:
like_fun = UnbinnedLikelihood(expectation_density)
cosi = ThreeMLPluginInterface('cosi', like_fun, response, bkg)

In [17]:
bkg_norm = bkg.norm.to_value(u.Hz)

cosi.bkg_parameter['bkg_norm'] = Parameter("bkg_norm",
                                          bkg_norm,
                                          unit = u.Hz,
                                          min_value=0,
                                          max_value=150,
                                          delta=0.05,
                                          )

print(f"The average background flux is {bkg_norm:.2f} Hz")

The average background flux is 25.61 Hz


In [18]:
plugins = DataList(cosi)
like = JointLikelihood(model, plugins, verbose=False) # You can enable debugging

21:53:15 INFO      set the minimizer to minuit                                             ]8;id=7713082;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=7713083;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

### Initializing the Cache

Load an existing NGC 4151 response cache when available.

In [19]:
cache_file = cache_path / "NGC_4151_source_response_cache.h5"
if cache_file.exists():
    response.load_caches(cache_path)
    print(f"Loaded response cache: {cache_file}")
else:
    print(f"No existing cache at {cache_file}; it will be computed.")

Loaded response cache: /home/parshap/COSI/Radio_Quiet_AGN/DC4/unbinned_cache/NGC_4151_source_response_cache.h5


The cache is initialized, which takes some time

In [20]:
print(f"Data Events: {data.nevents}\nExpected Events: {expectation_density.expected_counts():.2f}\nRelative Deviation {100 * (expectation_density.expected_counts()/data.nevents - 1):.3f} %")

21:54:53 INFO      Starting 3ML!                                  ]8;id=14478787;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=14478788;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\
         WARNING   WARNINGs here are NOT errors                   ]8;id=14478794;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=14478795;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\
         WARNING   but are inform you about optional packages     ]8;id=14478801;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=14478802;file:///home/parshap/miniforge3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#46\46]8;;\
                  that can be installed       

Evaluating the density:   0%|          | 0/25125951 [00:00<?, ?calls/s]

Data Events: 25125951
Expected Events: 25158429.42
Relative Deviation 0.129 %


Save the precomputed NGC 4151 response cache on Palmetto.

In [21]:
# response.save_caches(cache_path)
# print(f"Saved response cache: {cache_file}")

### Fitting

In [ ]:
print("Starting fit...", flush=True)
_ = like.fit()
print("Fit complete.", flush=True)

like.results.display()

results = like.results

parameters = {par.name: results.get_variates(par.path)
              for par in results.optimized_model["NGC_4151"].parameters.values()
              if par.free}

results_err = results.propagate(results.optimized_model["NGC_4151"].spectrum.main.shape.evaluate_at, **parameters)

energy = np.geomspace(100*u.keV, 10*u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux = results_err(e)
    flux_median[i] = flux.median
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)
    flux_inj[i] = spectrum_inj.evaluate_at(e)
    
%matplotlib inline

fig, ax = plt.subplots(figsize = (9, 6))

ax.plot(energy, flux_median, label = "Best fit")
ax.fill_between(energy, flux_lo, flux_hi, alpha = .5, label = "Best fit (errors)")
ax.plot(energy, flux_inj, color = 'black', ls = ":", label = "Injected")

ax.semilogx()
ax.semilogy()

ax.set_xlabel("Energy [keV]")
ax.set_ylabel(r"$\frac{\mathrm{d}N}{\mathrm{d}E}$ [keV$^{-1}$ cm$^{-2}$ s$^{-1}$]")

ax.legend();

plot_path = Path(
    "/home/parshap/cosipy/docs/tutorials/"
    "spectral_fits/continuum_fit/AGN/NGC4151_CPL_fit.pdf"
)
fig.savefig(
    plot_path,
    bbox_inches="tight",
    facecolor="white"
)

Starting fit...


In [ ]:
print("Starting fit...", flush=True)
_ = like.fit()
print("Fit complete.", flush=True)

like.results.display()

Now we can plot the result and compare it with the injected spectrum.

In [ ]:
results = like.results

parameters = {par.name: results.get_variates(par.path)
              for par in results.optimized_model["NGC_4151"].parameters.values()
              if par.free}

results_err = results.propagate(results.optimized_model["NGC_4151"].spectrum.main.shape.evaluate_at, **parameters)

In [24]:
energy = np.geomspace(100*u.keV, 10*u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux = results_err(e)
    flux_median[i] = flux.median
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)
    flux_inj[i] = spectrum_inj.evaluate_at(e)

NameError: name 'results_err' is not defined

In [ ]:
%matplotlib inline

In [23]:
fig, ax = plt.subplots(figsize = (9, 6))

ax.plot(energy, flux_median, label = "Best fit")
ax.fill_between(energy, flux_lo, flux_hi, alpha = .5, label = "Best fit (errors)")
ax.plot(energy, flux_inj, color = 'black', ls = ":", label = "Injected")

ax.semilogx()
ax.semilogy()

ax.set_xlabel("Energy [keV]")
ax.set_ylabel(r"$\frac{\mathrm{d}N}{\mathrm{d}E}$ [keV$^{-1}$ cm$^{-2}$ s$^{-1}$]")

ax.legend();

NameError: name 'energy' is not defined